# 12 — Every Version of a Table (Nessie Time Travel)

On Nessie, Iceberg's native `table.history()` collapses to a single snapshot because
each commit writes a fresh `metadata.json` that contains only the latest snapshot.
The real version log lives in **Nessie's commit history** — every Iceberg mutation
(`append`, `upsert`, `overwrite`, schema change) is a separate Nessie commit on `main`.

This notebook shows how to walk that history for *one specific table*, read the table
state at any past commit, and diff two versions.

Builds on notebook 08 (catalog refs) and assumes you've already produced multiple
commits — e.g. by running the weather or crypto ETL DAGs a few times.


In [1]:
import os
from datetime import datetime
from typing import Iterator

import polars as pl
import requests
from pyiceberg.catalog.rest import RestCatalog

NESSIE_URI       = os.environ["NESSIE_URI"]              # http://nessie:19120/iceberg
NESSIE_API       = NESSIE_URI.replace("/iceberg", "/api/v2")
S3_ENDPOINT      = os.environ["AWS_S3_ENDPOINT"]
S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]
WAREHOUSE_URI    = f"s3://{WAREHOUSE_BUCKET}/warehouse"

S3_PROPS = {
    "warehouse": WAREHOUSE_URI,
    "s3.endpoint": S3_ENDPOINT,
    "s3.access-key-id": S3_ACCESS_KEY,
    "s3.secret-access-key": S3_SECRET_KEY,
    "s3.path-style-access": "true",
    "s3.region": "us-east-1",
}

print("Nessie REST :", NESSIE_URI)
print("Nessie API  :", NESSIE_API)

Nessie REST : http://nessie:19120/iceberg
Nessie API  : http://nessie:19120/api/v2


## 1. Pick a target table

Change `TARGET` to any `(namespace, table)` tuple in your catalog. Defaults to
`bronze.weather_raw` because the weather DAG produces a commit per daily run.


In [2]:
TARGET = ("bronze", "weather_raw")
TABLE_FQN = f"{TARGET[0]}.{TARGET[1]}"
TABLE_FQN

'bronze.weather_raw'

## 2. List every Nessie commit that touched this table

Walk `/trees/main/history` with pagination and filter by `commitMeta.message`. Nessie
writes messages like `"Create ICEBERG_TABLE bronze.weather_raw"` and
`"Update ICEBERG_TABLE bronze.weather_raw"`, so a simple substring match is enough.


In [3]:
def iter_main_commits(page_size: int = 100, max_pages: int = 50) -> Iterator[dict]:
    """Yield every commitMeta dict on main, walking the pageToken until exhaustion."""
    params = {"maxRecords": page_size}
    for _ in range(max_pages):
        r = requests.get(f"{NESSIE_API}/trees/main/history", params=params)
        r.raise_for_status()
        body = r.json()
        for entry in body["logEntries"]:
            yield entry["commitMeta"]
        token = body.get("token")
        if not token:
            return
        params = {"maxRecords": page_size, "pageToken": token}


def commits_for_table(table_fqn: str) -> list[dict]:
    needle = f"ICEBERG_TABLE {table_fqn}"
    return [c for c in iter_main_commits() if needle in c.get("message", "")]


history = commits_for_table(TABLE_FQN)
print(f"Found {len(history)} commits touching {TABLE_FQN}\n")
for i, cm in enumerate(history[:15]):
    print(f"  [{i:>2}] {cm['hash'][:12]}  {cm['commitTime']}  {cm['message']}")

Found 12 commits touching bronze.weather_raw

  [ 0] 1ec4a339b7a9  2026-05-23T11:02:42.539062757Z  Update ICEBERG_TABLE bronze.weather_raw
  [ 1] 00ea9884c8f3  2026-05-23T11:02:26.542752180Z  Update ICEBERG_TABLE bronze.weather_raw
  [ 2] 5461b5cc73e3  2026-05-23T11:02:15.845271300Z  Update ICEBERG_TABLE bronze.weather_raw
  [ 3] 91a58dbd824c  2026-05-23T11:01:35.163001128Z  Update ICEBERG_TABLE bronze.weather_raw
  [ 4] fdecd50367a6  2026-05-23T10:57:37.198118671Z  Update ICEBERG_TABLE bronze.weather_raw
  [ 5] 8ba7c27fb0ad  2026-05-23T10:57:26.019980722Z  Update ICEBERG_TABLE bronze.weather_raw
  [ 6] 368c4a81e352  2026-05-23T10:56:46.792020051Z  Update ICEBERG_TABLE bronze.weather_raw
  [ 7] d323db7b5007  2026-05-23T10:56:32.049029669Z  Update ICEBERG_TABLE bronze.weather_raw
  [ 8] fb3cead54a1e  2026-05-23T10:55:31.059524126Z  Update ICEBERG_TABLE bronze.weather_raw
  [ 9] dcd222dd32ef  2026-05-23T10:55:21.954739553Z  Update ICEBERG_TABLE bronze.weather_raw
  [10] bf81b888f384  202

## 3. Read the table state at any past commit

Point a fresh `RestCatalog` at `{NESSIE_URI}/main@{hash}` — Nessie's URL grammar for
*"the `main` ref pinned to a specific commit"*. Each version's Iceberg `metadata.json`
is still in S3, just unreferenced by Iceberg's standard snapshot mechanism.


In [4]:
def catalog_at(commit_hash: str) -> RestCatalog:
    return RestCatalog(
        name=f"nessie@{commit_hash[:8]}",
        **{**S3_PROPS, "uri": f"{NESSIE_URI}/main@{commit_hash}"},
    )


def summarize_version(commit_hash: str, table_id: tuple[str, str]) -> dict:
    cat = catalog_at(commit_hash)
    try:
        t = cat.load_table(table_id)
        df = pl.from_arrow(t.scan().to_arrow())
        return {
            "hash": commit_hash[:12],
            "rows": df.height,
            "columns": len(t.schema().fields),
            "snapshot_id": t.current_snapshot().snapshot_id if t.current_snapshot() else None,
            "location": t.location().split("/warehouse/")[-1],
        }
    except Exception as e:
        return {"hash": commit_hash[:12], "rows": None, "columns": None,
                "snapshot_id": None, "location": f"<error: {type(e).__name__}>"}


# Sanity check: head of main
summarize_version(history[0]["hash"], TARGET) if history else "no commits yet"

{'hash': '1ec4a339b7a9',
 'rows': 264,
 'columns': 13,
 'snapshot_id': 7351079934310758371,
 'location': 'bronze/weather_raw_ae2180aa-1a78-4947-9647-f5ec8108796c'}

## 4. Walk every version into one table

This is the version log you actually want. One row per Nessie commit, in reverse
chronological order (latest first).


In [5]:
rows = []
for i, cm in enumerate(history):
    s = summarize_version(cm["hash"], TARGET)
    rows.append({
        "commit_idx": i,
        "commit_time": cm["commitTime"],
        "hash": s["hash"],
        "message": cm["message"],
        "rows": s["rows"],
        "cols": s["columns"],
        "snapshot_id": s["snapshot_id"],
    })

version_log = pl.DataFrame(rows)
version_log

commit_idx,commit_time,hash,message,rows,cols,snapshot_id
i64,str,str,str,i64,i64,i64
0,"""2026-05-23T11:02:42.539062757Z""","""1ec4a339b7a9""","""Update ICEBERG_TABLE bronze.we…",264,13,7351079934310758371
1,"""2026-05-23T11:02:26.542752180Z""","""00ea9884c8f3""","""Update ICEBERG_TABLE bronze.we…",240,13,2549601228501353997
2,"""2026-05-23T11:02:15.845271300Z""","""5461b5cc73e3""","""Update ICEBERG_TABLE bronze.we…",216,13,1868881171936492771
3,"""2026-05-23T11:01:35.163001128Z""","""91a58dbd824c""","""Update ICEBERG_TABLE bronze.we…",192,13,7596267586117503414
4,"""2026-05-23T10:57:37.198118671Z""","""fdecd50367a6""","""Update ICEBERG_TABLE bronze.we…",168,13,2950741286297060510
…,…,…,…,…,…,…
7,"""2026-05-23T10:56:32.049029669Z""","""d323db7b5007""","""Update ICEBERG_TABLE bronze.we…",96,13,6727912641296334357
8,"""2026-05-23T10:55:31.059524126Z""","""fb3cead54a1e""","""Update ICEBERG_TABLE bronze.we…",72,13,5741000150922061192
9,"""2026-05-23T10:55:21.954739553Z""","""dcd222dd32ef""","""Update ICEBERG_TABLE bronze.we…",48,13,4891444363665842195


## 5. Diff two versions

Read both versions, pick a logical key column, and compute *added / removed / shared*
row counts. For `bronze.weather_raw` the natural key is `(city, observation_ts)`; for
crypto bronze it's `(coin_id, ingest_ts)`. Adjust `KEY_COLS` below for other tables.

Defaults compare the latest commit against the one before it.


In [6]:
KEY_COLS = ["city", "observation_ts"]    # change per table
LEFT_IDX  = 1   # older
RIGHT_IDX = 0   # newer

def keys_at(commit_hash: str) -> set[tuple]:
    cat = catalog_at(commit_hash)
    t = cat.load_table(TARGET)
    df = pl.from_arrow(t.scan(selected_fields=tuple(KEY_COLS)).to_arrow())
    return set(map(tuple, df.iter_rows()))


if len(history) < 2:
    print("Need at least two commits to diff. Run the DAG a couple more times.")
else:
    left  = keys_at(history[LEFT_IDX]["hash"])
    right = keys_at(history[RIGHT_IDX]["hash"])
    added   = right - left
    removed = left  - right
    shared  = left  & right
    print(f"Older  ({history[LEFT_IDX]['hash'][:12]}, {history[LEFT_IDX]['commitTime']}): {len(left)} rows")
    print(f"Newer  ({history[RIGHT_IDX]['hash'][:12]}, {history[RIGHT_IDX]['commitTime']}): {len(right)} rows")
    print(f"  + added in newer   : {len(added)}")
    print(f"  - removed in newer : {len(removed)}")
    print(f"  = unchanged keys   : {len(shared)}")
    if added:
        print("  sample added :", list(added)[:3])
    if removed:
        print("  sample removed:", list(removed)[:3])

Older  (00ea9884c8f3, 2026-05-23T11:02:26.542752180Z): 48 rows
Newer  (1ec4a339b7a9, 2026-05-23T11:02:42.539062757Z): 48 rows
  + added in newer   : 0
  - removed in newer : 0
  = unchanged keys   : 48


## 6. Read the actual rows at a specific commit

Once you've identified an interesting commit from the version log, materialize the
full row set into Polars for inspection. This is the time-travel `SELECT *` you'd
want in Trino but can't get directly (see notebook 08 for why).


In [7]:
if history:
    pick_idx = min(2, len(history) - 1)         # 3rd most recent commit, or oldest if fewer
    pick_hash = history[pick_idx]["hash"]
    t = catalog_at(pick_hash).load_table(TARGET)
    df = pl.from_arrow(t.scan().to_arrow())
    print(f"Commit #{pick_idx} {pick_hash[:12]} — {history[pick_idx]['message']}")
    print(f"Rows: {df.height}, Columns: {df.width}")
    df.head(5)
else:
    print("No commits found for", TABLE_FQN)

Commit #2 5461b5cc73e3 — Update ICEBERG_TABLE bronze.weather_raw
Rows: 216, Columns: 13


## 7. Tag a commit so you can name it later

A Nessie *tag* is a named immutable pointer at a specific commit hash — handy for
marking "the version before we deployed the bad transformation". The tag is then
usable as a ref name in URIs: `{NESSIE_URI}/v_pre_fix`.

Re-runnable: deletes any pre-existing tag with the same name first.


In [8]:
TAG_NAME = "v_inspect_target"

if history:
    target_hash = history[min(2, len(history) - 1)]["hash"]

    existing = requests.get(f"{NESSIE_API}/trees/{TAG_NAME}")
    if existing.status_code == 200:
        h = existing.json()["reference"]["hash"]
        requests.delete(f"{NESSIE_API}/trees/{TAG_NAME}@{h}")

    main_ref = requests.get(f"{NESSIE_API}/trees/main").json()["reference"]
    create = requests.post(
        f"{NESSIE_API}/trees",
        params={"name": TAG_NAME, "type": "tag"},
        json={"type": "BRANCH", "name": "main", "hash": target_hash},
    )
    create.raise_for_status()
    print(f"Tag {TAG_NAME} → {target_hash[:12]}")

    # Read via the named tag instead of a raw hash
    via_tag = RestCatalog(
        name=f"nessie@{TAG_NAME}",
        **{**S3_PROPS, "uri": f"{NESSIE_URI}/{TAG_NAME}"},
    ).load_table(TARGET)
    print(f"Rows via tag {TAG_NAME}:", pl.from_arrow(via_tag.scan().to_arrow()).height)
else:
    print("No commits to tag.")

Tag v_inspect_target → 5461b5cc73e3
Rows via tag v_inspect_target: 216
